# الفصل الثاني عشر: الدوال المتقدمة (Lambda, Decorators, Generators, Closures)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-12.ipynb)

> **الوقت المتوقع:** ~9 دقيقة قراءة · **11** قسمًا · **41** مثالًا برمجيًا.

**كيف تستخدم هذا الدفتر؟**

1. شغّل الخلايا بالترتيب من الأعلى إلى الأسفل (`Shift` + `Enter`).
2. خلايا الشيفرة قابلة للتعديل: غيّر القيم وأعد التشغيل لتجرّب.
3. خلايا 🧾 معروضة للقراءة فقط (أوامر طرفية، أو إدخال يدوي، أو مقتطف ناقص)، وخلايا ⚠️ تُظهر خطأً مقصودًا يُلتقط تلقائيًا فلا يتوقف «تشغيل الكل».

> **Kaggle:** نزّل الملف من المستودع ثم `File ▸ Import Notebook`، أو استورده مباشرة من GitHub بعد ربط حسابك.

<details>
<summary><b>محتويات الفصل</b></summary>

- **12.1** لماذا هذا الفصل هو نقطة التحوّل نحو الاحتراف؟
- **12.2** دوال Lambda: الدوال المجهولة
- **12.3** الإغلاقات (Closures): الدوال التي تتذكر
- **12.4** المزخرفات (Decorators): تعديل سلوك الدوال
- **12.5** المولّدات (Generators): الكفاءة في الذاكرة
- **12.6** جدول مقارنة شامل: المفاهيم الأربعة
- **12.7** مشروع تطبيقي: نظام مراقبة أداء مع Decorators وGenerators
- **12.8** تمارين
- **12.9** اختبر فهمك (Quiz)
- **12.10** ملخص الفصل (Cheat Sheet)
- **12.11** ماذا بعد؟

</details>

In [ ]:
#@title ⚙️ تهيئة البيئة  { display-mode: "form" }
import sys, os, random, pathlib

print("Python:", sys.version.split()[0], "|", sys.platform)
assert sys.version_info >= (3, 10), "هذا الكتاب يفترض Python 3.10 أو أحدث"

SEED = 42
random.seed(SEED)

## 12.1 لماذا هذا الفصل هو نقطة التحوّل نحو الاحتراف؟

إذا فتحت الكود المصدري لأي مكتبة Python شهيرة — Flask، FastAPI، Django، Pytest — ستجد هذه الأربعة مفاهيم في كل مكان: `@app.route(...)`، `@pytest.fixture`، `yield`، دوال `lambda` صغيرة كمعاملات. هذه ليست «حيل متقدمة» للمحترفين فقط؛ إنها **اللغة الأساسية** التي يتحدث بها كود Python الحديث.

حتى الآن، تعاملت مع الدوال كـ«آلات»: تُدخل مُدخلات، تحصل على مُخرجات. في هذا الفصل، ستكتشف أن الدوال في Python أقوى بكثير من ذلك:
- يمكنك كتابة دالة **صغيرة جداً وسريعة** دون حتى تسميتها (Lambda)
- يمكنك **تعديل سلوك دالة موجودة** دون تعديل كودها الأصلي (Decorators)
- يمكنك كتابة دالة **تتذكر مكانها** بين الاستدعاءات، وتنتج قيماً واحدة تلو الأخرى بكفاءة هائلة في الذاكرة (Generators)
- يمكنك جعل دالة «تحمل معها» بيانات من بيئتها حتى بعد انتهاء تلك البيئة (Closures)

هذه المفاهيم الأربعة مترابطة، وستبني كل واحدة على سابقتها. لنبدأ.

## 12.2 دوال Lambda: الدوال المجهولة

### الفكرة الأساسية

**Lambda** هي دالة صغيرة «مجهولة» (بدون اسم)، تُكتب في سطر واحد، ومُصممة للاستخدام السريع مرة واحدة أو كمعامل لدالة أخرى.

In [ ]:
# دالة عادية
def square(x: int) -> int:
    return x ** 2

# نفس الدالة بصيغة lambda
square_lambda = lambda x: x ** 2

print(square(5))         # 25
print(square_lambda(5))  # 25

**الصيغة العامة**: `lambda parameters: expression`

> ⚠️ **قيود مهمة على lambda**:
> - جسمها **تعبير واحد فقط** (Expression)، وليس سلسلة من الجُمل (Statements)
> - لا يمكن أن تحتوي على `if/else` كجُمل كاملة (لكن يمكن استخدام Ternary Expression)
> - لا تحتوي `return` صراحة — القيمة الناتجة من التعبير تُعاد تلقائياً

In [ ]:
# lambda مع عدة معاملات
add = lambda a, b: a + b
print(add(3, 4))  # 7

# lambda مع Ternary Expression
is_even = lambda n: "زوجي" if n % 2 == 0 else "فردي"
print(is_even(7))  # فردي

# lambda بدون معاملات
greet = lambda: "مرحباً!"
print(greet())  # مرحباً!

### متى تُستخدم lambda فعلياً؟

> ⚠️ **تحذير احترافي مهم**: لا تُسند دالة lambda لمتغير كما في `square_lambda = lambda x: x**2` في الكود الحقيقي! هذا يُعتبر ممارسة سيئة (حتى أن PEP 8 ينص على تجنبها صراحة). إذا احتجت لدالة لها اسم، استخدم `def` العادية:

```python
# ❌ ممارسة سيئة - استخدم def بدلاً منها
square = lambda x: x ** 2

# ✅ أفضل
def square(x: int) -> int:
    return x ** 2
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

**الاستخدام الصحيح والشائع لـ lambda**: كمعامل مؤقت «يُستهلك فوراً» داخل دالة أخرى، دون الحاجة لتسميته:

In [ ]:
students = [
    {"name": "Ahmed", "score": 85},
    {"name": "Sara", "score": 92},
    {"name": "Omar", "score": 78}
]

# مع sorted() - الاستخدام الأشهر لـ lambda
sorted_by_score = sorted(students, key=lambda s: s["score"], reverse=True)
for s in sorted_by_score:
    print(s["name"], s["score"])

# مع min() و max()
youngest_scorer = min(students, key=lambda s: s["score"])
print(youngest_scorer)

> 🔄 **مقارنة**: لماذا لا نكتب دالة `def get_score(s): return s["score"]` بدلاً من lambda؟ كلاهما صحيح! لكن عندما تكون المنطق بسيطاً جداً (سطر واحد) ولن يُستخدم إلا هنا فقط، lambda توفر إيجازاً دون تلويث مساحة الاسم (Namespace) بدالة صغيرة لن تُستخدم مجدداً. القاعدة العملية: **إذا كانت الدالة ستُستخدم أكثر من مرة، أو منطقها معقد قليلاً، استخدم `def`. إذا كانت بسيطة واستخدامها لمرة واحدة فقط كمعامل، استخدم lambda.**

### Lambda مع map() و filter()

In [ ]:
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# map(): تطبيق دالة على كل عنصر
squared = list(map(lambda x: x ** 2, numbers))
print(squared)  # [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]

# filter(): تصفية العناصر حسب شرط
evens = list(filter(lambda x: x % 2 == 0, numbers))
print(evens)  # [2, 4, 6, 8, 10]

> 🔄 **مقارنة - أيهما أفضل: map/filter أم List Comprehension؟**

In [ ]:
# بـ map/filter
result = list(map(lambda x: x ** 2, filter(lambda x: x % 2 == 0, numbers)))

# بـ List Comprehension - الأسلوب الـPythonic الموصى به
result = [x ** 2 for x in numbers if x % 2 == 0]

معظم خبراء Python (بمن فيهم مُنشئ اللغة نفسه Guido van Rossum) يُفضّلون **List Comprehension** على `map/filter` لأنها أوضح وأقرب للغة الطبيعية. لكن من المهم معرفة `map` و`filter` لأنك ستجدهما كثيراً في كود الآخرين، وفي بعض الحالات (خصوصاً مع دوال موجودة مسبقاً وليس lambda) تكون `map` مفيدة وواضحة:

In [ ]:
# استخدام map مع دالة موجودة - مقبول تماماً وشائع
str_numbers = ["1", "2", "3"]
int_numbers = list(map(int, str_numbers))
print(int_numbers)  # [1, 2, 3]

## 12.3 الإغلاقات (Closures): الدوال التي تتذكر

### الدوال المتداخلة (Nested Functions)

قبل فهم Closures، لنفهم أن Python تسمح بتعريف دالة **داخل** دالة أخرى:

In [ ]:
def outer_function():
    message = "مرحباً من outer!"

    def inner_function():
        print(message)  # inner تصل لمتغير outer بفضل قاعدة LEGB (الفصل 10)

    inner_function()

outer_function()  # مرحباً من outer!

### ما هو الـ Closure فعلياً؟

**Closure** يحدث عندما تُرجع دالة داخلية من دالة خارجية، وتبقى الدالة الداخلية «تتذكر» المتغيرات من بيئة الدالة الخارجية، **حتى بعد انتهاء تنفيذ الدالة الخارجية**!

In [ ]:
def make_multiplier(factor: int):
    """تُرجع دالة تضرب أي رقم بـ factor."""
    def multiplier(number: int) -> int:
        return number * factor  # factor "مُغلَقة" داخل هذه الدالة
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(5))   # 10
print(triple(5))   # 15

**ماذا حدث هنا بالضبط؟**
1. `make_multiplier(2)` تُنفَّذ، فيُنشأ متغير محلي `factor = 2`
2. تُعرَّف دالة `multiplier` داخلياً، وتُستخدم `factor` بداخلها
3. `make_multiplier` تُعيد `multiplier` **كقيمة** (بدون استدعائها)، وتنتهي
4. منطقياً، يجب أن يختفي `factor` الآن لأن الدالة انتهت... **لكنه لا يختفي!**
5. Python تحتفظ بـ`factor` «محبوسة» داخل الدالة المُعادة، لأن `multiplier` ما زالت تحتاجها

هذا يُسمى **Closure**: الدالة الداخلية «تُغلِق» على المتغيرات من بيئتها الخارجية.

In [ ]:
# تأكيد عملي أن كل closure يحتفظ بنسخته الخاصة
print(double(10))  # 20 - لا تزال تتذكر factor=2
print(triple(10))  # 30 - لا تزال تتذكر factor=3

### استخدام `nonlocal` لتعديل متغير من البيئة الخارجية

In [ ]:
def make_counter():
    """تُرجع دالة تعد تصاعدياً في كل استدعاء."""
    count = 0

    def increment():
        nonlocal count  # بدون هذه الكلمة، سيحدث UnboundLocalError
        count += 1
        return count

    return increment

counter1 = make_counter()
print(counter1())  # 1
print(counter1())  # 2
print(counter1())  # 3

counter2 = make_counter()  # عداد جديد مستقل تماماً
print(counter2())  # 1  -- لا علاقة له بـ counter1!

> 💡 **نصيحة الخبراء**: `nonlocal` تشبه `global` (الفصل 10)، لكنها تشير لمتغير في **الدالة المحيطة (Enclosing)** وليس المستوى العام (Global) للملف. استخدمها فقط عندما تحتاج لتعديل متغير من البيئة الخارجية داخل closure.

### تطبيق عملي: Closures لإنشاء دوال مخصصة

In [ ]:
def make_validator(min_length: int):
    """تُنشئ دالة تحقق مخصصة بحد أدنى معين."""
    def validate(text: str) -> bool:
        return len(text) >= min_length
    return validate

validate_username = make_validator(5)
validate_password = make_validator(8)

print(validate_username("ali"))      # False (3 أحرف فقط)
print(validate_username("ahmed99"))  # True
print(validate_password("short"))    # False
print(validate_password("secure123"))  # True

> 🔄 **مقارنة مع البرمجة الكائنية**: Closures هي في الحقيقة طريقة «خفيفة الوزن» لتحقيق ما تحققه الكائنات (Objects) عادة: بيانات مرتبطة بسلوك. سنرى في الفصل 13 أن الـ Classes توفر نفس الفكرة لكن بقدرات أكبر عندما تحتاج حالة معقدة متعددة الخصائص. القاعدة العملية: استخدم Closures للحالات البسيطة (متغير أو اثنين)، واستخدم Classes عندما تكبر الحاجة.

## 12.4 المزخرفات (Decorators): تعديل سلوك الدوال

### الفكرة الأساسية

**Decorator** هو دالة تأخذ دالة أخرى كمُدخل، وتُرجع دالة جديدة «تُحيط» بسلوك الدالة الأصلية — تُضيف سلوكاً قبل و/أو بعد تنفيذها، **دون تعديل كود الدالة الأصلية نفسه**.

لنبني Decorator خطوة بخطوة لفهم آلية عمله الداخلية:

In [ ]:
def my_decorator(func):
    """decorator بسيط يطبع رسالة قبل وبعد تنفيذ الدالة."""
    def wrapper():
        print("🔵 قبل تنفيذ الدالة...")
        func()  # استدعاء الدالة الأصلية
        print("🔴 بعد تنفيذ الدالة...")
    return wrapper

def say_hello():
    print("مرحباً!")

# التطبيق اليدوي (بدون @)
decorated_hello = my_decorator(say_hello)
decorated_hello()

**الناتج:**

```
🔵 قبل تنفيذ الدالة...
مرحباً!
🔴 بعد تنفيذ الدالة...
```

### الصيغة الأنيقة: `@`

Python توفر صياغة خاصة (Syntactic Sugar) لتطبيق الـ decorator مباشرة:

In [ ]:
def my_decorator(func):
    def wrapper():
        print("🔵 قبل تنفيذ الدالة...")
        func()
        print("🔴 بعد تنفيذ الدالة...")
    return wrapper

@my_decorator  # هذا يُعادل: say_hello = my_decorator(say_hello)
def say_hello():
    print("مرحباً!")

say_hello()  # نفس النتيجة السابقة تماماً!

`@my_decorator` فوق `def say_hello():` هي ببساطة اختصار لـ `say_hello = my_decorator(say_hello)`.

### Decorators مع دوال لها معاملات

المشكلة: دالة `wrapper` أعلاه لا تقبل أي معاملات. لجعل الـ decorator يعمل مع **أي** دالة بغض النظر عن معاملاتها، نستخدم `*args` و`**kwargs`:

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print(f"🔵 استدعاء {func.__name__} بالمعاملات: {args}, {kwargs}")
        result = func(*args, **kwargs)
        print(f"🔴 النتيجة: {result}")
        return result
    return wrapper

@my_decorator
def add(a: int, b: int) -> int:
    return a + b

result = add(3, 5)

**الناتج:**

```
🔵 استدعاء add بالمعاملات: (3, 5), {}
🔴 النتيجة: 8
```

> ⚠️ **خطأ شائع**: نسيان `return result` في `wrapper`، مما يجعل أي دالة مُزخرفة تُرجع `None` دائماً بغض النظر عن قيمتها الحقيقية:

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    def bad_decorator(func):
        def wrapper(*args, **kwargs):
            func(*args, **kwargs)  # ⚠️ نُفِّذت لكن النتيجة ضاعت!
            # لا يوجد return هنا!
        return wrapper

    @bad_decorator
    def add(a, b):
        return a + b

    print(add(3, 5))  # None  💥 بدلاً من 8!
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

### الحفاظ على معلومات الدالة الأصلية: `functools.wraps`

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def greet(name: str) -> str:
    """تُرجع تحية للمستخدم."""
    return f"مرحباً {name}"

print(greet.__name__)  # wrapper  ⚠️ فقدنا الاسم الأصلي!
print(greet.__doc__)   # None     ⚠️ فقدنا التوثيق الأصلي!

**الحل**:

In [ ]:
from functools import wraps

def my_decorator(func):
    @wraps(func)  # يحافظ على __name__ و __doc__ الأصليين
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def greet(name: str) -> str:
    """تُرجع تحية للمستخدم."""
    return f"مرحباً {name}"

print(greet.__name__)  # greet  ✅
print(greet.__doc__)   # تُرجع تحية للمستخدم.  ✅

> 💡 **نصيحة الخبراء - قاعدة ذهبية**: **كل decorator تكتبه يجب أن يحتوي `@wraps(func)`** فوق تعريف `wrapper`. هذا جزء أساسي من «النمط الصحيح» لكتابة decorators في Python، ونسيانه يُسبب مشاكل غامضة لاحقاً في الأدوات التي تعتمد على `__name__` مثل أنظمة التوثيق التلقائي وبعض أطر العمل.

### Decorators عملية حقيقية

**1. قياس زمن التنفيذ:**

In [ ]:
import time
from functools import wraps

def measure_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print(f"⏱️ {func.__name__} استغرقت {end - start:.4f} ثانية")
        return result
    return wrapper

@measure_time
def slow_calculation(n: int) -> int:
    total = sum(i ** 2 for i in range(n))
    return total

result = slow_calculation(1_000_000)
print(result)

**2. تسجيل الاستدعاءات (Logging):**

In [ ]:
from functools import wraps

def log_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"📝 استُدعيت: {func.__name__}({args}, {kwargs})")
        result = func(*args, **kwargs)
        print(f"📝 أُرجعت: {result}")
        return result
    return wrapper

@log_calls
def divide(a: float, b: float) -> float:
    return a / b

divide(10, 2)

**3. إعادة المحاولة عند الفشل (Retry):**

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    from functools import wraps
    import random

    def retry(max_attempts: int = 3):
        """Decorator factory: تُرجع decorator حقيقياً حسب max_attempts."""
        def decorator(func):
            @wraps(func)
            def wrapper(*args, **kwargs):
                for attempt in range(1, max_attempts + 1):
                    try:
                        return func(*args, **kwargs)
                    except Exception as e:
                        print(f"⚠️ محاولة {attempt} فشلت: {e}")
                        if attempt == max_attempts:
                            raise
                return None
            return wrapper
        return decorator

    @retry(max_attempts=3)
    def unreliable_operation():
        if random.random() < 0.7:
            raise ConnectionError("فشل الاتصال!")
        return "✅ نجحت العملية!"

    print(unreliable_operation())
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

> 💡 **ملاحظة مهمة**: `retry(max_attempts=3)` هذا نمط يُسمى **Decorator Factory** — دالة تُرجع decorator، مما يسمح بتمرير معاملات مخصصة للـ decorator نفسه. لاحظ وجود 3 مستويات من الدوال المتداخلة: `retry` → `decorator` → `wrapper`.

**4. التحقق من الصلاحيات (نموذج شائع في تطوير الويب):**

```python
from functools import wraps

def require_login(func):
    @wraps(func)
    def wrapper(user: dict, *args, **kwargs):
        if not user.get("is_logged_in"):
            print("❌ يجب تسجيل الدخول أولاً.")
            return None
        return func(user, *args, **kwargs)
    return wrapper

@require_login
def view_profile(user: dict) -> None:
    print(f"👤 ملف {user['name']} الشخصي")

guest = {"name": "زائر", "is_logged_in": False}
logged_in_user = {"name": "أحمد", "is_logged_in": True}

view_profile(guest)          # ❌ يجب تسجيل الدخول أولاً.
view_profile(logged_in_user)  # 👤 ملف أحمد الشخصي
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

### تطبيق عدة decorators معاً

In [ ]:
@measure_time
@log_calls
def process(n: int) -> int:
    return n ** 2

process(10)

> ⚠️ **تحذير مهم**: ترتيب الـ decorators مهم! تُطبَّق **من الأسفل للأعلى**. الكود أعلاه يُعادل: `process = measure_time(log_calls(process))`. أي أن `log_calls` تُطبَّق أولاً (الأقرب للدالة)، ثم `measure_time` تُحيط بالنتيجة بالكامل.

### Decorators مدمجة شهيرة في Python

سنتعمق في هذه في الفصل 13 و14، لكن إليك لمحة سريعة لأنها decorators حقيقية تستخدم نفس المبدأ:

In [ ]:
class Circle:
    def __init__(self, radius: float):
        self._radius = radius

    @property  # decorator مدمج - يجعل method يُستخدم كخاصية
    def area(self) -> float:
        return 3.14159 * self._radius ** 2

c = Circle(5)
print(c.area)  # 78.53975 -- بدون أقواس استدعاء!

from functools import lru_cache

@lru_cache(maxsize=None)  # تذكر - caching تلقائي! (كتبناه يدوياً في الفصل 10)
def fibonacci(n: int) -> int:
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print(fibonacci(30))  # سريع جداً بفضل lru_cache، رغم أن fibonacci بطيئة أصلاً

> 💡 **نصيحة الخبراء**: `@lru_cache` هي نفس فكرة دالة `memoize` التي بنيتها يدوياً في تمرين الفصل العاشر، لكنها جاهزة ومُحسّنة من المكتبة القياسية. استخدمها دائماً بدلاً من كتابة caching يدوي، إلا إذا احتجت سلوكاً مخصصاً جداً.

## 12.5 المولّدات (Generators): الكفاءة في الذاكرة

### المشكلة: القوائم الكبيرة تستهلك ذاكرة هائلة

In [ ]:
def get_squares(n: int) -> list[int]:
    """تُنشئ قائمة كاملة بكل المربعات - تستهلك ذاكرة كبيرة!"""
    return [i ** 2 for i in range(n)]

# إذا n = 10,000,000، هذا ينشئ قائمة ضخمة بالكامل في الذاكرة فوراً!
squares = get_squares(10_000_000)

ماذا لو كنت تحتاج فقط لمعالجة هذه الأرقام واحداً تلو الآخر، دون الحاجة لتخزينها كلها في نفس اللحظة؟ هنا يأتي دور **Generators**.

### إنشاء Generator بـ `yield`

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    def get_squares_generator(n: int):
        """تُنتج المربعات واحداً تلو الآخر، دون تخزين كل شيء في الذاكرة."""
        for i in range(n):
            yield i ** 2  # yield بدلاً من return

    gen = get_squares_generator(5)
    print(gen)  # <generator object get_squares_generator at 0x...>

    print(next(gen))  # 0
    print(next(gen))  # 1
    print(next(gen))  # 4
    print(next(gen))  # 9
    print(next(gen))  # 16
    # print(next(gen))  # 💥 StopIteration - انتهت القيم
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

**الفرق الجوهري بين `return` و`yield`**:
- `return` تُنهي الدالة فوراً وتُعيد قيمة واحدة
- `yield` «تُجمّد» الدالة في مكانها، تُعيد قيمة واحدة، **وتتذكر مكانها بالضبط** لتكمل من هناك عند الاستدعاء التالي

In [ ]:
# غالباً نستخدم for مباشرة بدلاً من next() اليدوية
for square in get_squares_generator(5):
    print(square)
# 0
# 1
# 4
# 9
# 16

### توضيح آلية العمل خطوة بخطوة

In [ ]:
def demo_generator():
    print("البداية")
    yield 1
    print("بعد العنصر الأول")
    yield 2
    print("بعد العنصر الثاني")
    yield 3
    print("النهاية")

gen = demo_generator()  # لا شيء يُطبع بعد! الدالة لم تبدأ التنفيذ بعد

print("--- أول next ---")
print(next(gen))  # البداية ثم 1

print("--- ثاني next ---")
print(next(gen))  # بعد العنصر الأول ثم 2

print("--- ثالث next ---")
print(next(gen))  # بعد العنصر الثاني ثم 3

**الناتج:**

```
--- أول next ---
البداية
1
--- ثاني next ---
بعد العنصر الأول
2
--- ثالث next ---
بعد العنصر الثاني
3
```

> 💡 هذا يكشف حقيقة مهمة: مجرد استدعاء `demo_generator()` **لا يُنفّذ أي كود فعلياً**! الدالة تبدأ التنفيذ فقط عند أول `next()`، وتتوقف عند كل `yield`، منتظرة الطلب التالي.

### Generator Expressions: نسخة مختصرة من Comprehension

In [ ]:
# List Comprehension - ينشئ القائمة بالكامل فوراً في الذاكرة
squares_list = [x ** 2 for x in range(1_000_000)]

# Generator Expression - ينتج القيم عند الطلب فقط (lazy evaluation)
squares_gen = (x ** 2 for x in range(1_000_000))

print(type(squares_list))  # <class 'list'>
print(type(squares_gen))   # <class 'generator'>

> 🔄 **مقارنة الذاكرة - دليل عملي**:

In [ ]:
import sys

squares_list = [x ** 2 for x in range(1_000_000)]
squares_gen = (x ** 2 for x in range(1_000_000))

print(f"حجم القائمة: {sys.getsizeof(squares_list):,} بايت")
print(f"حجم المولّد: {sys.getsizeof(squares_gen):,} بايت")

**الناتج التقريبي:**

```
حجم القائمة: 8,448,728 بايت  (حوالي 8 ميجابايت!)
حجم المولّد: 200 بايت         (ثابت بغض النظر عن n!)
```

هذا هو السبب الجوهري لاستخدام Generators: **كفاءة هائلة في الذاكرة** عند التعامل مع بيانات كبيرة أو تدفقات لا نهائية.

> ⚠️ **تحذير مهم**: Generator يُستهلك **مرة واحدة فقط**! بعد المرور على كل عناصره، لا يمكن إعادة استخدامه:

In [ ]:
gen = (x for x in range(3))

print(list(gen))  # [0, 1, 2]
print(list(gen))  # []  ⚠️ فارغة! تم استهلاك المولّد بالكامل

إذا احتجت المرور على البيانات أكثر من مرة، استخدم `list()` أو أعد إنشاء الـ generator من جديد.

### متى تستخدم Generator بدلاً من List؟

| الحالة | الخيار الأفضل |
|---|---|
| تحتاج كل العناصر في الذاكرة فعلياً للوصول العشوائي (`lst[5]`) | List |
| تحتاج المرور على البيانات أكثر من مرة | List |
| البيانات ضخمة جداً أو لا نهائية | Generator |
| تحتاج فقط المرور مرة واحدة بالترتيب | Generator |
| معالجة ملفات ضخمة سطراً سطراً | Generator |
| سلسلة عمليات متتالية (Pipeline) على بيانات كبيرة | Generator |

### تطبيق عملي: معالجة ملف ضخم

In [ ]:
def read_large_file(file_path: str):
    """تقرأ ملفاً ضخماً سطراً سطراً دون تحميله بالكامل في الذاكرة."""
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            yield line.strip()

def count_lines_with_keyword(file_path: str, keyword: str) -> int:
    """تحسب عدد الأسطر المحتوية على كلمة معينة، بكفاءة ذاكرة عالية."""
    count = 0
    for line in read_large_file(file_path):
        if keyword in line:
            count += 1
    return count

# حتى لو كان الملف بحجم عدة جيجابايت، هذا الكود يستهلك ذاكرة ثابتة تقريباً!
# count = count_lines_with_keyword("huge_log.txt", "ERROR")

### تمرين توضيحي: سلسلة Fibonacci لا نهائية

```python
def infinite_fibonacci():
    """مولّد لا نهائي لمتتالية فيبوناتشي."""
    a, b = 0, 1
    while True:  # حلقة لا نهائية - آمنة هنا لأن المولّد lazy!
        yield a
        a, b = b, a + b

fib = infinite_fibonacci()
first_10 = [next(fib) for _ in range(10)]
print(first_10)
# [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
```

⚠️ **للقراءة فقط** — حلقة تنتظر إدخالًا من المستخدم.

> 💡 **نصيحة الخبراء**: لاحظ أن `while True` هنا **آمنة تماماً**! في الفصل السابع حذّرنا من الحلقات اللا نهائية، لكن داخل generator، الحلقة تتوقف عند كل `yield` ولا تُكمل أبداً إلا عند طلب القيمة التالية صراحة. هذا نمط شائع جداً ومفيد لإنتاج تدفقات بيانات لا نهائية نظرياً (مثل قراءة بيانات حساسة بالوقت، أو تسلسل أرقام).

### دالة `itertools`: صندوق أدوات Generators المتقدم

```python
from itertools import islice, count, cycle

# islice: أخذ جزء محدد من generator لا نهائي
fib = infinite_fibonacci()
first_5 = list(islice(fib, 5))
print(first_5)  # [0, 1, 1, 2, 3]

# count: عداد لا نهائي يبدأ من رقم معين
for i in islice(count(10, 2), 5):  # يبدأ من 10، بخطوة 2
    print(i, end=" ")
print()  # 10 12 14 16 18

# cycle: تكرار عناصر قائمة بشكل لا نهائي
colors = cycle(["red", "green", "blue"])
for color in islice(colors, 7):
    print(color, end=" ")
print()  # red green blue red green blue red
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `infinite_fibonacci` من مثال سابق.

## 12.6 جدول مقارنة شامل: المفاهيم الأربعة

| المفهوم | الهدف الأساسي | متى تستخدمه |
|---|---|---|
| **Lambda** | دالة مجهولة سريعة الكتابة | كمعامل مؤقت لـ `sorted`, `map`, `filter`, وما شابه |
| **Closure** | دالة تتذكر بيانات من بيئتها | إنشاء دوال «مُخصّصة» بإعدادات ثابتة (factories) |
| **Decorator** | تعديل سلوك دالة دون تعديل كودها | Logging, قياس الوقت, التحقق من الصلاحيات, Caching |
| **Generator** | إنتاج قيم بكفاءة ذاكرة عالية | بيانات ضخمة, تدفقات لا نهائية, معالجة ملفات كبيرة |

## 12.7 مشروع تطبيقي: نظام مراقبة أداء مع Decorators وGenerators

سنبني نظاماً يدمج كل مفاهيم الفصل: decorator لقياس الأداء وتسجيل الاستدعاءات، وgenerator لمعالجة بيانات ضخمة بكفاءة.

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    import time
    import random
    from functools import wraps

    # --- جزء 1: Decorators للمراقبة ---

    call_log: list[dict] = []

    def monitor(func):
        """decorator يسجل كل استدعاء: الاسم، الوقت المستغرق، النتيجة."""
        @wraps(func)
        def wrapper(*args, **kwargs):
            start = time.perf_counter()
            try:
                result = func(*args, **kwargs)
                status = "نجاح"
            except Exception as e:
                result = str(e)
                status = "فشل"
                raise
            finally:
                duration = time.perf_counter() - start
                call_log.append({
                    "function": func.__name__,
                    "duration": round(duration, 6),
                    "status": status,
                })
            return result
        return wrapper

    def retry(max_attempts: int = 3, delay: float = 0.1):
        """decorator factory لإعادة المحاولة عند الفشل."""
        def decorator(func):
            @wraps(func)
            def wrapper(*args, **kwargs):
                last_error = None
                for attempt in range(1, max_attempts + 1):
                    try:
                        return func(*args, **kwargs)
                    except Exception as e:
                        last_error = e
                        time.sleep(delay)
                raise last_error
            return wrapper
        return decorator

    # --- جزء 2: دوال العمل الفعلية ---

    @monitor
    @retry(max_attempts=3)
    def fetch_data_from_source(source_id: int) -> dict:
        """تحاكي جلب بيانات من مصدر خارجي، قد يفشل أحياناً."""
        if random.random() < 0.3:
            raise ConnectionError(f"فشل الاتصال بالمصدر {source_id}")
        return {"source_id": source_id, "value": random.randint(1, 100)}

    # --- جزء 3: Generator لمعالجة البيانات بكفاءة ---

    def data_source_generator(count: int):
        """يولّد معرفات مصادر بيانات واحداً تلو الآخر."""
        for i in range(1, count + 1):
            yield i

    def process_all_sources(count: int) -> list[dict]:
        """يعالج كل المصادر، مستخدماً generator لتوفير الذاكرة."""
        results = []
        for source_id in data_source_generator(count):
            try:
                data = fetch_data_from_source(source_id)
                results.append(data)
            except ConnectionError as e:
                print(f"❌ فشل نهائي: {e}")
        return results

    # --- جزء 4: تقرير الأداء ---

    def print_performance_report() -> None:
        print("\n" + "=" * 50)
        print("📊 تقرير مراقبة الأداء")
        print("=" * 50)

        total_calls = len(call_log)
        successful = sum(1 for log in call_log if log["status"] == "نجاح")
        total_time = sum(log["duration"] for log in call_log)

        print(f"إجمالي الاستدعاءات: {total_calls}")
        print(f"الناجحة: {successful} | الفاشلة: {total_calls - successful}")
        print(f"إجمالي الوقت: {total_time:.4f} ثانية")

        if call_log:
            avg_time = total_time / total_calls
            print(f"متوسط الوقت لكل استدعاء: {avg_time:.6f} ثانية")
        print("=" * 50)

    # --- التشغيل ---
    if __name__ == "__main__":
        results = process_all_sources(10)

        print(f"\n✅ تم جلب {len(results)} نتيجة بنجاح:")
        for r in results:
            print(f"  مصدر {r['source_id']}: قيمة = {r['value']}")

        print_performance_report()
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

**مثال ناتج (قد يختلف بسبب العشوائية):**

```
✅ تم جلب 10 نتيجة بنجاح:
  مصدر 1: قيمة = 42
  مصدر 2: قيمة = 17
  ...

==================================================
📊 تقرير مراقبة الأداء
==================================================
إجمالي الاستدعاءات: 13
الناجحة: 10 | الفاشلة: 3
إجمالي الوقت: 0.3201 ثانية
متوسط الوقت لكل استدعاء: 0.024623 ثانية
==================================================
```

<details>
<summary>✅ تحدي إضافي: إضافة decorator لتحديد السرعة (Rate Limiting) - الحل</summary>

In [ ]:
import time
from functools import wraps

def rate_limit(calls_per_second: float):
    """decorator factory يمنع استدعاء الدالة أسرع من معدل معين."""
    min_interval = 1.0 / calls_per_second
    last_called = [0.0]  # قائمة بعنصر واحد لتجنب مشاكل nonlocal في closures متداخلة

    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            elapsed = time.perf_counter() - last_called[0]
            wait_time = min_interval - elapsed
            if wait_time > 0:
                time.sleep(wait_time)
            last_called[0] = time.perf_counter()
            return func(*args, **kwargs)
        return wrapper
    return decorator

@rate_limit(calls_per_second=2)  # بحد أقصى استدعائين في الثانية
def api_call(n: int) -> str:
    return f"نتيجة الاستدعاء {n}"

start = time.perf_counter()
for i in range(5):
    print(api_call(i))
print(f"الوقت الإجمالي: {time.perf_counter() - start:.2f} ثانية")  # ≈ 2 ثانية

</details>

## 12.8 تمارين

### 🟢 مستوى مبتدئ

**تمرين 1**: استخدم `lambda` مع `sorted()` لترتيب قائمة `["banana", "kiwi", "apple", "fig"]` حسب طول الكلمة.

**تمرين 2**: اكتب generator باسم `countdown(n)` يُنتج الأرقام من n إلى 1 تنازلياً.

<details>
<summary>✅ الحل</summary>

In [ ]:
# تمرين 1
fruits = ["banana", "kiwi", "apple", "fig"]
sorted_fruits = sorted(fruits, key=lambda word: len(word))
print(sorted_fruits)  # ['fig', 'kiwi', 'apple', 'banana']

# تمرين 2
def countdown(n: int):
    while n > 0:
        yield n
        n -= 1

for num in countdown(5):
    print(num, end=" ")  # 5 4 3 2 1

</details>

### 🟡 مستوى متوسط

**تمرين 3**: اكتب decorator باسم `uppercase_result` يحوّل أي نتيجة نصية من دالة إلى أحرف كبيرة تلقائياً.

**تمرين 4**: استخدم Closure لكتابة دالة `make_bank_account(initial_balance)` تُرجع دالتين: `deposit(amount)` و`check_balance()`، بحيث يبقى الرصيد محفوظاً بين الاستدعاءات.

<details>
<summary>✅ الحل</summary>

In [ ]:
# تمرين 3
from functools import wraps

def uppercase_result(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

@uppercase_result
def get_greeting(name: str) -> str:
    return f"hello, {name}"

print(get_greeting("ahmed"))  # HELLO, AHMED

# تمرين 4
def make_bank_account(initial_balance: float):
    balance = initial_balance

    def deposit(amount: float) -> None:
        nonlocal balance
        balance += amount
        print(f"✅ تم إيداع {amount}. الرصيد الآن: {balance}")

    def check_balance() -> float:
        return balance

    return deposit, check_balance

deposit, check_balance = make_bank_account(1000)
deposit(500)                      # ✅ تم إيداع 500. الرصيد الآن: 1500
print(check_balance())            # 1500
deposit(200)                      # ✅ تم إيداع 200. الرصيد الآن: 1700
print(check_balance())            # 1700

</details>

### 🔴 تحدي

**تمرين 5**: اكتب generator باسم `batch_generator(items, batch_size)` يأخذ قائمة ويُنتج «دفعات» (قوائم فرعية) بحجم `batch_size` في كل مرة. مثال: `batch_generator([1,2,3,4,5,6,7], 3)` يُنتج `[1,2,3]` ثم `[4,5,6]` ثم `[7]`.

<details>
<summary>✅ الحل</summary>

In [ ]:
def batch_generator(items: list, batch_size: int):
    """يقسّم قائمة إلى دفعات، مفيد عند معالجة بيانات ضخمة على أجزاء."""
    batch = []
    for item in items:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:  # إرسال أي عناصر متبقية أقل من batch_size
        yield batch

items = [1, 2, 3, 4, 5, 6, 7]
for batch in batch_generator(items, 3):
    print(batch)

# [1, 2, 3]
# [4, 5, 6]
# [7]

> 💡 هذا النمط شائع جداً في معالجة البيانات الضخمة (مثل إرسال طلبات API بدفعات لتجنب تجاوز حدود المعدل) وفي تدريب نماذج الذكاء الاصطناعي (تقسيم البيانات إلى batches).
</details>

## 12.9 اختبر فهمك (Quiz)

<details>
<summary>❓ السؤال 1: ما الفرق بين return وyield؟</summary>

**الإجابة**: `return` تُنهي تنفيذ الدالة فوراً وتُعيد قيمة واحدة نهائية. `yield` «تُجمّد» حالة الدالة عند تلك النقطة، تُعيد قيمة واحدة، لكنها تتذكر مكانها بالضبط (كل المتغيرات المحلية) لتُكمل التنفيذ من حيث توقفت عند الاستدعاء التالي لـ `next()`. دالة تحتوي `yield` تُصبح تلقائياً "Generator Function".
</details>

<details>
<summary>❓ السؤال 2: لماذا يجب استخدام @wraps(func) داخل كل decorator؟</summary>

**الإجابة**: بدون `@wraps(func)`، تفقد الدالة المُزخرفة هويتها الأصلية — `__name__` يصبح `"wrapper"` بدلاً من اسم الدالة الحقيقي، و`__doc__` يصبح `None` بدلاً من التوثيق الأصلي. هذا يُسبب مشاكل في التصحيح (Debugging) والتوثيق التلقائي، وقد يكسر أدوات تعتمد على معرفة اسم الدالة الحقيقي.
</details>

<details>
<summary>❓ السؤال 3: ما ناتج الكود التالي؟</summary>

In [ ]:
def make_adder(x):
    def add(y):
        return x + y
    return add

add5 = make_adder(5)
add10 = make_adder(10)
print(add5(3))
print(add10(3))

**الإجابة**: `8` ثم `13`. هذا مثال على Closure: كل استدعاء لـ `make_adder` يُنشئ دالة `add` جديدة «تتذكر» قيمة `x` الخاصة بها بشكل مستقل تماماً عن الاستدعاءات الأخرى.
</details>

<details>
<summary>❓ السؤال 4: لماذا تُعتبر Generators أكثر كفاءة في الذاكرة من القوائم؟</summary>

**الإجابة**: القائمة تُنشئ وتُخزّن **كل** عناصرها في الذاكرة فور إنشائها. الـ Generator يُنتج عنصراً واحداً فقط عند الطلب (`next()`)، ولا يحتفظ بالعناصر السابقة أو اللاحقة في الذاكرة، مما يجعل حجمه في الذاكرة ثابتاً تقريباً بغض النظر عن عدد العناصر المحتملة، حتى لو كانت لا نهائية.
</details>

<details>
<summary>❓ السؤال 5: متى يجب استخدام lambda بدلاً من def؟</summary>

**الإجابة**: عندما تحتاج دالة بسيطة جداً (تعبير واحد) تُستخدم لمرة واحدة فقط، عادة كمعامل لدالة أخرى مثل `sorted()`, `map()`, أو `filter()`. لا يُفضّل إسناد lambda لمتغير باسم دائم؛ في تلك الحالة `def` أوضح وأفضل دائماً.
</details>

## 12.10 ملخص الفصل (Cheat Sheet)

```python
# ============ LAMBDA ============
square = lambda x: x ** 2              # ⚠️ تجنّب تسميتها؛ استخدم def
sorted(data, key=lambda x: x["score"]) # ✅ الاستخدام الصحيح

# ============ CLOSURE ============
def outer(x):
    def inner(y):
        return x + y   # inner "تُغلِق" على x
    return inner

add5 = outer(5)
add5(3)  # 8

# nonlocal لتعديل متغير البيئة الخارجية
def counter():
    count = 0
    def increment():
        nonlocal count
        count += 1
        return count
    return increment

# ============ DECORATOR ============
from functools import wraps

def my_decorator(func):
    @wraps(func)               # ✅ دائماً!
    def wrapper(*args, **kwargs):
        # كود قبل
        result = func(*args, **kwargs)
        # كود بعد
        return result           # ✅ لا تنسَ return!
    return wrapper

@my_decorator
def my_function(): ...

# Decorator Factory (مع معاملات)
def repeat(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(times=3)
def greet(): ...

# ============ GENERATOR ============
def my_generator():
    yield 1
    yield 2
    yield 3

for value in my_generator():
    print(value)

# Generator Expression
gen = (x ** 2 for x in range(1000000))  # lazy - كفاءة ذاكرة عالية

# itertools
from itertools import islice, count, cycle
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `data` من مثال سابق.

| المفهوم | الكلمة المفتاحية | الاستخدام الأمثل |
|---|---|---|
| Lambda | `lambda` | معامل سريع لـ sorted/map/filter |
| Closure | `nonlocal` | دوال «مُخصّصة» تحمل حالة بسيطة |
| Decorator | `@`, `functools.wraps` | Logging, Timing, Caching, Auth |
| Generator | `yield` | بيانات ضخمة، تدفقات، كفاءة ذاكرة |

## 12.11 ماذا بعد؟

لقد انتهيت الآن من **الجزء الثاني بالكامل تقريباً**: البرمجة الوظيفية المتقدمة. تمتلك الآن أدوات يستخدمها كبار مبرمجي Python يومياً، وستراها منتشرة في كل مكتبة احترافية تتعامل معها لاحقاً.

لكن حتى الآن، كل برامجنا نظّمت البيانات والسلوك **بشكل منفصل**: بيانات في قواميس وقوائم، وسلوك في دوال مستقلة. ماذا لو أردنا تجميع البيانات **والسلوك المرتبط بها** في «شيء واحد» متكامل؟ ماذا لو أردنا إنشاء «قالب» نصنع منه كائنات متعددة، كل واحد له بياناته الخاصة لكن بنفس السلوك؟

هذا هو جوهر **البرمجة الكائنية (Object-Oriented Programming - OOP)** — النموذج الذي بُنيت عليه أغلب أنظمة البرمجيات الكبرى في العالم، من تطبيقات الويب إلى الألعاب إلى أنظمة التشغيل.

في **الفصل الثالث عشر: البرمجة الكائنية - الأساس**، ستتعلم `class`, `object`, `__init__`, و`self` — بداية رحلة ستغيّر جذرياً طريقة تفكيرك في بناء البرامج.

---

### الخطوة التالية

- [الفصل الثالث عشر: البرمجة الكائنية - الأساس (Class, Object, __init__, self)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-13.ipynb)
- [فهرس كل الدفاتر](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/00-index.ipynb)

*الفصل الثاني عشر — من كتاب «بايثون 3.14: من الصفر إلى الاحتراف».*